# 🐕 Module 1: Foundations of Vision and Image Analysis
## Fine-Grained Dog Breed Recognition Project

This notebook implements low-level image processing techniques.

**Instructions:** Upload 5 diverse dog images when prompted:
- 1 Fluffy dog (e.g., Samoyed, Pomeranian)
- 1 Dark dog (e.g., Rottweiler, Doberman)
- 1 Light dog (e.g., Golden Retriever)
- 1 Smooth coat dog (e.g., Beagle, Boxer)
- 1 Long-haired dog (e.g., Afghan Hound)

---
## Step 1: Setup Environment

In [ ]:
# Install required packages
!pip install opencv-python-headless matplotlib numpy scikit-image -q

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import os

# Display helper functions
def show_images_row(images, titles, figsize=(20, 4)):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if len(img.shape) == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("✅ Setup complete!")

---
## Step 2: Upload Your 5 Dog Images

**Click the button below and select 5 images from your local dataset.**

Pick diverse dogs: fluffy, dark, light, smooth, long-haired.

In [ ]:
# Upload images from your local machine
print("📤 Please select 5 dog images from your local dataset...")
print("(Pick diverse types: fluffy, dark, light, smooth, long-haired)\n")

uploaded = files.upload()

print(f"\n✅ Uploaded {len(uploaded)} files!")

In [ ]:
# Load uploaded images
sample_images = {}
display_size = (250, 250)

# Assign labels based on upload order
labels = ['Dog 1', 'Dog 2', 'Dog 3', 'Dog 4', 'Dog 5']

for i, (filename, content) in enumerate(uploaded.items()):
    # Decode image from bytes
    nparr = np.frombuffer(content, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    if img is not None:
        label = labels[i] if i < len(labels) else f'Dog {i+1}'
        sample_images[label] = {
            'image': img, 
            'filename': filename
        }
        print(f"✅ {label}: {filename} - Shape: {img.shape}")
    else:
        print(f"❌ Could not load: {filename}")

print(f"\nTotal images loaded: {len(sample_images)}")

In [ ]:
# Optional: Rename your dogs for clarity
# Uncomment and edit the lines below to give meaningful names

# new_names = {
#     'Dog 1': 'Fluffy (Samoyed)',
#     'Dog 2': 'Dark (Rottweiler)',
#     'Dog 3': 'Light (Golden)',
#     'Dog 4': 'Smooth (Beagle)',
#     'Dog 5': 'Long-haired (Afghan)'
# }
# sample_images = {new_names.get(k, k): v for k, v in sample_images.items()}

# Display all uploaded images
images = [cv2.resize(data['image'], display_size) for data in sample_images.values()]
titles = list(sample_images.keys())

print("🐕 Your 5 Dog Images:")
show_images_row(images, titles, figsize=(20, 5))

---
## Step 3: Geometric Transformations

In [ ]:
# Geometric transform functions
def rotate_image(image, angle):
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, matrix, (w, h), borderMode=cv2.BORDER_REFLECT)

def scale_image(image, factor):
    h, w = image.shape[:2]
    return cv2.resize(image, (int(w * factor), int(h * factor)))

def flip_image(image):
    return cv2.flip(image, 1)

# Apply to all dogs
print("📐 GEOMETRIC TRANSFORMATIONS")
print("="*60)

for dog_type, data in sample_images.items():
    img = data['image']
    transforms = [
        ('Original', img),
        ('Rotate 30°', rotate_image(img, 30)),
        ('Scale 0.7x', scale_image(img, 0.7)),
        ('Flip', flip_image(img))
    ]
    images = [cv2.resize(t[1], display_size) for t in transforms]
    titles = [t[0] for t in transforms]
    
    print(f"\n{dog_type}:")
    show_images_row(images, titles, figsize=(16, 4))

---
## Step 4: Intensity Transformations

In [ ]:
# Intensity transform functions
def histogram_equalization(image, method='clahe'):
    if len(image.shape) == 3:
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l_channel = lab[:, :, 0]
    else:
        l_channel = image
    
    if method == 'global':
        equalized = cv2.equalizeHist(l_channel)
    else:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        equalized = clahe.apply(l_channel)
    
    if len(image.shape) == 3:
        lab[:, :, 0] = equalized
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return equalized

def gamma_correction(image, gamma):
    table = np.array([(i / 255.0) ** (1.0 / gamma) * 255 for i in range(256)]).astype(np.uint8)
    return cv2.LUT(image, table)

# Apply to all dogs
print("🌓 INTENSITY TRANSFORMATIONS")
print("="*60)

for dog_type, data in sample_images.items():
    img = data['image']
    transforms = [
        ('Original', img),
        ('CLAHE', histogram_equalization(img, 'clahe')),
        ('Gamma 0.5 (Bright)', gamma_correction(img, 0.5)),
        ('Gamma 2.0 (Dark)', gamma_correction(img, 2.0))
    ]
    images = [cv2.resize(t[1], display_size) for t in transforms]
    titles = [t[0] for t in transforms]
    
    print(f"\n{dog_type}:")
    show_images_row(images, titles, figsize=(16, 4))

---
## Step 5: Edge Detection

In [ ]:
# Edge detection functions
def sobel_edges(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return cv2.convertScaleAbs(np.sqrt(sobel_x**2 + sobel_y**2))

def canny_edges(image, low=50, high=150):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.4)
    return cv2.Canny(blurred, low, high)

def compute_edge_density(edge_img):
    return np.sum(edge_img > 0) / edge_img.size

# Apply to all dogs
print("🔲 EDGE DETECTION")
print("="*60)

edge_densities = {}
for dog_type, data in sample_images.items():
    img = data['image']
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    edges = [
        ('Grayscale', gray),
        ('Sobel', sobel_edges(img)),
        ('Canny (Fine)', canny_edges(img, 30, 90)),
        ('Canny (Coarse)', canny_edges(img, 100, 200))
    ]
    images = [cv2.resize(e[1], display_size) for e in edges]
    titles = [e[0] for e in edges]
    
    density = compute_edge_density(canny_edges(img, 50, 150))
    edge_densities[dog_type] = density
    
    print(f"\n{dog_type} - Edge Density: {density:.2%}")
    show_images_row(images, titles, figsize=(16, 4))

print("\n📊 EDGE DENSITY RANKING:")
for dtype, density in sorted(edge_densities.items(), key=lambda x: x[1], reverse=True):
    print(f"  {dtype}: {density:.2%}")

---
## Step 6: Noise & Restoration

In [ ]:
# Noise functions
def add_gaussian_noise(image, sigma=25):
    noise = np.random.normal(0, sigma, image.shape).astype(np.float32)
    noisy = image.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def add_salt_pepper_noise(image, prob=0.02):
    noisy = image.copy()
    rnd = np.random.random(image.shape[:2])
    noisy[rnd < prob/2] = 255
    noisy[(rnd >= prob/2) & (rnd < prob)] = 0
    return noisy

# Restoration filters
def median_filter(image, ksize=5):
    return cv2.medianBlur(image, ksize)

def bilateral_filter(image):
    return cv2.bilateralFilter(image, 9, 75, 75)

def nlm_denoise(image):
    return cv2.fastNlMeansDenoisingColored(image, None, 10, 10, 7, 21)

# PSNR metric
def calculate_psnr(original, processed):
    mse = np.mean((original.astype(np.float64) - processed.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10((255 ** 2) / mse)

print("✅ Noise and restoration functions ready!")

In [ ]:
# Apply noise and restoration
print("🔊 NOISE & RESTORATION")
print("="*60)

psnr_results = []

for dog_type, data in sample_images.items():
    img = data['image']
    noisy = add_gaussian_noise(img, sigma=25)
    
    restorations = [
        ('Original', img),
        ('Noisy', noisy),
        ('Median', median_filter(noisy)),
        ('Bilateral', bilateral_filter(noisy)),
        ('NLM', nlm_denoise(noisy))
    ]
    images = [cv2.resize(r[1], display_size) for r in restorations]
    titles = [r[0] for r in restorations]
    
    print(f"\n{dog_type}:")
    show_images_row(images, titles, figsize=(20, 4))
    
    # PSNR values
    psnr_noisy = calculate_psnr(img, noisy)
    psnr_median = calculate_psnr(img, median_filter(noisy))
    psnr_bilateral = calculate_psnr(img, bilateral_filter(noisy))
    psnr_nlm = calculate_psnr(img, nlm_denoise(noisy))
    
    print(f"  PSNR: Noisy={psnr_noisy:.1f}dB | Median={psnr_median:.1f}dB | "
          f"Bilateral={psnr_bilateral:.1f}dB | NLM={psnr_nlm:.1f}dB")
    
    psnr_results.append({
        'dog': dog_type,
        'noisy': psnr_noisy,
        'median': psnr_median,
        'bilateral': psnr_bilateral,
        'nlm': psnr_nlm
    })

In [ ]:
# PSNR Summary Table
print("\n" + "="*70)
print("📊 PSNR COMPARISON TABLE (Higher = Better)")
print("="*70)
print(f"{'Dog':<15} {'Noisy':>10} {'Median':>10} {'Bilateral':>10} {'NLM':>10}")
print("-"*70)

for r in psnr_results:
    print(f"{r['dog']:<15} {r['noisy']:>10.2f} {r['median']:>10.2f} "
          f"{r['bilateral']:>10.2f} {r['nlm']:>10.2f}")

print("="*70)

---
## Step 7: Summary

In [ ]:
print("="*70)
print("🎓 MODULE 1 COMPLETE!")
print("="*70)
print("""
KEY FINDINGS:

📌 GEOMETRIC TRANSFORMS:
   - Rotation, scaling, flipping preserve image content
   - Useful for data augmentation in Module 3

📌 INTENSITY TRANSFORMS:
   - CLAHE improves contrast while preserving texture
   - Dark dogs benefit most from histogram equalization

📌 EDGE DETECTION:
   - Fluffy dogs → Higher edge density (more texture)
   - Smooth dogs → Lower edge density

📌 NOISE & RESTORATION:
   - Bilateral filter: Best overall (edge-preserving)
   - Median filter: Best for salt-and-pepper noise
   - NLM: Highest quality but slower

RECOMMENDED PIPELINE:
   Input → CLAHE → Bilateral Filter → Ready for Module 2
""")
print("="*70)
print("Next: Module 2 (SIFT, HOG, LBP + SVM Classification)")